# Modelo Propio

Para poder hacer un modelo funcional, si o si, tenemos que usar CNN, que son redes neuronales, estas dan un mayor resultado que algoritmos tradicionales.

Estas nos dan un mayor resultado, porque las capas iniciales detectan los bordes, esquinas y texturas.

Las capas de un nivel intermedio, son capas más profundas que reconocen objetos, por ejemplo ruedas.

Por último las capas de alto nivel, ensamblan partes para formar objetos completos.

In [ ]:
#Hacemos login en huggin face, para poder acceder a los datasets
from huggingface_hub import notebook_login
notebook_login()

# Todos los imports necesarios

In [ ]:
#Carga del dataset hugging face
from datasets import load_dataset
#Descarga de las imagenes S3
import requests
from PIL import Image
from io import BytesIO
import tensorflow as tf
import os
import time
import pandas as pd
import re

tf.random.set_seed(42)

## Almacenamiento de las imagenes

Para poder entrenar al modelo, necesitamos tener esas imagenes descargadas y separadas por carpetas, para que se puedan splitear bien con Tensorflow para entregarselas al modelo.

Para ello en la variable de abajo, tienes que definir una ruta en tu drive, con la ubicación de donde quieres que se cree ese dataset de imagenes.

Tenemos que usar tensorflow, porque **keras** tiene una herramienta maestra, que de la estructura de carpetas que hemos creado, nos crea un dataframe y etiqueta directamente las imagenes, de manera optimizada y lista para la CNN

In [ ]:
driveURL = '/content/drive/MyDrive/Proyecto/dataset'

In [ ]:
ds = load_dataset("juppy44/gbif-plants-raw", split="train", streaming=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.88k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

# Problema principal

Las imagenes las tienen en un S3 de amazon, por ende no podemos pasarselo al modelo.

1. Tenemos que descargar imagene a imagen y asociarla a su nombre cientifico
2. Modificar esta imagen para que tenga 224x224 que es lo que admite el modelo
3. Entrenarlo.

Para ello tenemos diferentes opciones
* Usando requests y Pillow
* Boto3 (librería de amazon integrada con S3)

## transformResolution(image, weigth, heigth)


*   image  - ImageFile
*   weigth - Integer
*   heigth - Integer
*   return ImageFile

Esta función se encarga de coger una Imagen descargada del bucket S3 de iNaturalist y redimensionarla al tamaño que necesita nuestro modelo para poder entrenarlo.


In [ ]:
def transformResolution(image, weigth=224, heigth=224):
  return image.resize((weigth, heigth))

## clearFolderName(name)

 Limpia y valida un nombre para usarlo como carpeta o archivo.
    
    Args:
        name (str): Nombre original que puede contener caracteres inválidos
    
    Returns:
        str: Nombre limpio o None si el input no es válido
    
    Example:
        >>> clearFolderName("A/B:C?D*E")
        'A_B_C_D_E'
        >>> clearFolderName(None)
        None




In [ ]:
def clearFolderName(name):
  if name is None or pd.isna(name):
    return None

  name = str(name).strip()
  if name == "" or name.lower() in ["na", "nan", "null", "none", "undefined"]:
    return None

  # Remover caracteres no válidos
  invalid_chars = r'[<>:"/\\|?*]'
  name = re.sub(invalid_chars, '_', name)

  # Limitar longitud
  if len(name) > 50:
    name = name[:50]

  return name

## getBetterLabel(row)

Determina la mejor etiqueta disponible basándose en la información taxonómica.
    
    Prioridad de campos:
    1. species_name (más específico)
    2. genus_name
    3. family_name
    4. dataset_name + basis_of_record (contexto)
    5. "unknown" (último recurso)
    
    Args:
        row (dict): Fila del dataset con información taxonómica
    
    Returns:
        str: Etiqueta óptima para organizar la imagen
    
    Example:
        >>> row = {"species_name": "Canis lupus", "genus_name": "Canis"}
        >>> getBetterLabel(row)
        'Canis lupus'

In [ ]:
def getBetterLabel(row):

  # Orden de prioridad: species -> genus -> family
  for field in ["species_name", "genus_name", "family_name"]:
    value = clearFolderName(row.get(field))
    if value:
      if field == "species_name":
        return value
      elif field == "genus_name":
        return f"genus_{value}"
      else:
        return f"family_{value}"


## generateFilename(row, urlS3, speciesName)
   
    Genera un nombre de archivo único basado en metadatos.
    
    Estrategias en orden de prioridad:

    - Usar timestamp + especie (evita colisiones)

    
    Args:
        speciesName (str): Nombre de la especie/categoría
    
    Returns:
        str: Nombre de archivo único con extensión .jpg


In [ ]:
def generateFilename(speciesName):
  timestamp = int(time.time() * 1000)  # Milisegundos para mayor unicidad
  species_short = speciesName[:30]  # Acortar para evitar nombres muy largos
  return f"{timestamp}_{species_short}.jpg"

## downloadImages(urlS3, row, basePath)
Descarga una imagen desde una URL y la guarda organizada por taxonomía.
    
    Flujo:
    1. Determinar etiqueta óptima usando getBetterLabel()
    2. Crear estructura de carpetas
    3. Descargar y validar imagen
    4. Redimensionar imagen a 224x224
    5. Guardar con nombre único
    
    Args:
        urlS3 (str): URL de la imagen a descargar
        row (dict): Fila completa del dataset con todos los metadatos
        basePath (str): Ruta base donde se guardarán las imágenes
    
    Returns:
        str or None: Ruta del archivo guardado o None si hubo error
    
    Raises:
        requests.exceptions.RequestException: Error en la descarga HTTP
        IOError: Error al procesar la imagen
    
    Example:
        >>> row = dataset[0]
        >>> downloadImages(row["image_url"], row, "./data")
        './data/Canis_lupus/123456_Canis_lupus.jpg'

In [ ]:
def downloadImages (urlS3, row, basePath):
  try:
    # 1. Validar URL
    if not urlS3 or pd.isna(urlS3):
      print("URL vacía o inválida, saltando descarga.")
      return None

    # 2. Obtener la mejor etiqueta disponible
    speciesName = getBetterLabel(row)
    if not speciesName:
      speciesName = "unknown"

    # 3. Crear estructura de carpetas
    speciesFolder = os.path.join(basePath, speciesName)
    os.makedirs(speciesFolder, exist_ok=True)

    # 4. Descargar el contenido de la URL
    request = requests.get(urlS3, timeout=10)
    request.raise_for_status()

    # 5. Procesar imagen
    imageVanilla = Image.open(BytesIO(request.content))
    imageResize = transformResolution(imageVanilla)

    # 6. Generar nombre de archivo único
    filename = generateFilename(speciesName)
    filepath = os.path.join(speciesFolder, filename)

    # 7. Guardar imagen
    if imageResize.mode != 'RGB':
      imageResize = imageResize.convert('RGB')

    imageResize.save(filepath, 'JPEG', quality=85)

  except requests.exceptions.RequestException as e:
    print(f"Error al descargar la imagen: {e}")
    return None
  except IOError as e:
    print(f"Error al procesar la imagen: {e}")
    return None
  except Exception as e:
    print(f"Error inesperado: {e}")
    return None


# Elección del tipo de CNN

**Red Neuronal Residual - ResNet-50**

Hemos seleccionado la arquitectura ResNet-50 (Red Residual de 50 capas) desarrollada por He et al. (2016) por las siguientes razones:

### 1. **Arquitectura Especializada para Profundidad**
A diferencia de las redes secuenciales tradicionales, ResNet-50 incorpora **conexiones residuales (skip connections)** que permiten el flujo directo de información entre capas no adyacentes. Esto resuelve el problema del *vanishing gradient* en redes profundas, permitiendo entrenar eficientemente sus 50 capas.

### 2. **Optimizada para Clasificación de Imágenes**
ResNet-50 fue específicamente diseñada y optimizada para el desafío ImageNet, alcanzando un **76.1% de precisión Top-1** con 90 épocas de entrenamiento. Su estructura en bloques residuales (cada uno con convoluciones 1×1, 3×3, 1×1) permite aprender representaciones jerárquicas complejas esenciales para distinguir entre miles de categorías visuales.

### 3. **Superioridad sobre Redes Secuenciales**
Mientras que una red secuencial de 50 capas sufriría de degradación en el aprendizaje, ResNet-50 mantiene un entrenamiento estable gracias a su fórmula residual:  
`y = F(x) + x` donde `x` es la entrada y `F(x)` la transformación aprendida.

### 4. **Transferibilidad Comprobada**
Las características aprendidas por ResNet-50 en datasets generales como ImageNet han demostrado alta transferibilidad a dominios específicos, incluyendo la botánica. Esto es crucial para nuestro proyecto, donde planeamos utilizar tanto entrenamiento desde cero como fine-tuning de modelos pre-entrenados.

## Algoritmo de Visión Computacional

Hemos escogido el RestNet-50 ya que viendo sus [benchmarks](https://github.com/tensorflow/models/blob/master/official/vision/MODEL_GARDEN.md) con los demás es el que menos computo necesita (lo cual nos favorece) y muestra unos resultados bastante atractivos.

Es un modelo de Tensorflow.

###Resultados del benchmark de las CNN:
| Modelo      | Resolución | Épocas | Top-1 (%) | Top-5 (%) | Observaciones |
|-------------|------------|--------|-----------|-----------|---------------|
| ResNet-50   | 224×224    | 90     | 76.1      | 92.9      | Configuración básica |
| ResNet-50   | 224×224    | 200    | 77.1      | 93.5      | +1.0% Top-1 con más épocas |
| ResNet-101  | 224×224    | 200    | 78.3      | 94.2      | Más parámetros → +1.2% vs ResNet-50 |
| ResNet-152  | 224×224    | 200    | 78.7      | 94.3      | Mayor capacidad (+0.4% vs ResNet-101) |


Como vemos en el propio **benchmark** tenemos que transformar todas las imagenes antes de guardarlas, estas deben tener una resolución de 224x224, para que el modelo pueda trabajar con nuestros datos

# Estudio del dataset que tenemos
El dataset proviene de [Huggin face](https://huggingface.co/)

Los datos que vamos a usar para evaluar el modelo, son los mismos que tiene iNaturalis de un modelo pre-entrenado. [dataset](https://huggingface.co/datasets/juppy44/gbif-plants-raw/viewer/default/train)

In [ ]:
print("Estudio de los datos")
print(f"Total de columnas: {len(ds.column_names)}")
print(f"Nombre de las columnas y tipo de dato ")
for col_name, feature_type in ds.features.items():
    print(f"   • {col_name}: {feature_type}")

Estudio de los datos
Total de columnas: 17
Nombre de las columnas y tipo de dato 
   • gbif_id: Value('string')
   • species_id: Value('string')
   • genus_id: Value('string')
   • family_id: Value('string')
   • species_name: Value('string')
   • genus_name: Value('string')
   • family_name: Value('string')
   • lat: Value('float64')
   • lon: Value('float64')
   • event_date: Value('string')
   • dataset_key: Value('string')
   • dataset_name: Value('string')
   • basis_of_record: Value('string')
   • image_url: Value('string')
   • license_raw: Value('string')
   • rights_holder: Value('string')
   • day_of_year: Value('int64')


Tenemos 17 columnas, pero realmente no nos interesan todas. Las columnas más importantes son:
1.  **image_url**, esta nos permite descargarnos una imagen de la especie, a la que luego tendremos que añadirle datos, rotandola, porque solo ofrece un tipo.
2. **species_name**: Esta nos indica que tipo de especie es, el problema es que para algunas plantas tiene valur null
3. **genus_name**: Este nos indica el nombre cientifico, para el caso en el que alguna planta no tenga nombre de especie, podemos sacarlo con su nombre cientifico para buscar sus cuidados.
4. **family_name**: no es tan relevente, pero en el caso de que nos falle el species_name y genus_name, podemos obtener una orientación de como cuidar una planta por este campo, porque es la familia que agrupa varios genus_name.

Para entrenar una red neuronal convolucional CNN, tenemos que saber cuantas clases de plantas tenemos primero, para ello realizamos esta función:

In [ ]:
def countSpecies(dataset, maxRows , columnTarget = "species_name" ):
  if columnTarget not in dataset.column_names:
          print(f"❌ Columna '{columnTarget}' no encontrada")
          print(f"   Columnas disponibles: {dataset.column_names}")
          return None

  classes = set()
  nulls = 0
  total = 0

  for row in dataset.take(maxRows):
    total = total + 1
    if row["species_name"] is None:
      nulls = nulls + 1
    if row["species_name"] not in classes:
      classes.add(row["species_name"])

  print("=" * 50)
  print("RESULTADOS DE ANALISIS DE CLASES")
  print("=" * 50)

  print(f"Cantidad de especies con valo nulo: {nulls}")
  print(f"Muestras analizadas: {total}")
  print(f"Clases únicas: {len(classes)}")



In [ ]:
countSpecies(ds, 500)

RESULTADOS DE ANALISIS DE CLASES
Cantidad de especies con valo nulo: 6
Muestras analizadas: 500
Clases únicas: 272


In [ ]:
nImages = 10
for row in ds.take(nImages):
  print(row["species_id"], row["image_url"])
  downloadImages(row["image_url"], row, driveURL)

3152226 https://inaturalist-open-data.s3.amazonaws.com/photos/562214896/original.jpg
3170266 https://inaturalist-open-data.s3.amazonaws.com/photos/562362130/original.jpg
7300476 https://inaturalist-open-data.s3.amazonaws.com/photos/27393808/original.jpg
7300476 https://inaturalist-open-data.s3.amazonaws.com/photos/27393806/original.jpg
7300476 https://inaturalist-open-data.s3.amazonaws.com/photos/27393803/original.jpg
6391648 https://inaturalist-open-data.s3.amazonaws.com/photos/562781833/original.jpg
8346401 https://inaturalist-open-data.s3.amazonaws.com/photos/16145728/original.jpg
8346401 https://inaturalist-open-data.s3.amazonaws.com/photos/16145761/original.jpg
2641920 https://inaturalist-open-data.s3.amazonaws.com/photos/16544399/original.jpg
5275012 https://inaturalist-open-data.s3.amazonaws.com/photos/16705094/original.jpg
